In [4]:
# Name: Muhammad Arif Luqman bin Mohd Zaihan, Muhammad Rayyan Farhan bin Zuhardi
# ID: (SW01083752), (SW01083761)
# Course: CISB5123 Text Analytics

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_mal_reviews(base_url, max_pages=5):
    """
    Scrapes reviewer name, date, and content from MyAnimeList reviews.
    Navigates through pagination up to the specified max_pages limit.
    """
    reviews_data = []
    
    # Standard User-Agent header to prevent the site from blocking the request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Loop through the pages (1 to max_pages)
    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        
        # Append the page number to the URL query string
        url = f"{base_url}?p={page}"
        response = requests.get(url, headers=headers)
        
        # Check if the page loaded successfully
        if response.status_code != 200:
            print(f"Failed to retrieve page {page}. Status code: {response.status_code}")
            break
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all individual review containers on the page
        review_elements = soup.find_all('div', class_='review-element')
        
        if not review_elements:
            print("No more reviews found. Ending scrape early.")
            break
            
        # Extract data from each review container
        for review in review_elements:
            # Extract Reviewer Name
            name_tag = review.find('div', class_='username')
            name = name_tag.text.strip() if name_tag else "Anonymous"
            
            # Extract Review Date
            date_tag = review.find('div', class_='update_at')
            date = date_tag.text.strip() if date_tag else "Unknown Date"
            
            # Extract Review Content
            content_tag = review.find('div', class_='text')
            content = content_tag.text.strip() if content_tag else "No content"
            
            # Append the extracted info to our list
            reviews_data.append({
                "Reviewer Name": name,
                "Review Date": date,
                "Review Content": content
            })
            
        # Pause for 2 seconds between pages to be polite to the servers
        time.sleep(2)
        
    return reviews_data

def save_to_csv(data, filename="one_piece_reviews.csv"):
    """
    Converts the list of review dictionaries into a Pandas DataFrame
    and saves it as a CSV file.
    """
    if not data:
        print("No data was extracted to save.")
        return
        
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Data successfully saved to {filename}")

# --- Main Execution Block ---
if __name__ == "__main__":
    # Target URL specifically for the One Piece anime
    target_url = "https://myanimelist.net/anime/21/One_Piece/reviews"
    
    # Execute the scraper with the lab-required limit of 5 pages
    scraped_data = scrape_mal_reviews(target_url, max_pages=5)
    
    # Save the extracted data to a properly named file
    save_to_csv(scraped_data, "one_piece_reviews.csv")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Data successfully saved to one_piece_reviews.csv
